In [ ]:
#| default_exp classify_by_activity

# Tile-activity classification

> Per-HiRISE-observation classification of tile marking-count distributions into **bimodal** (busy/not-busy two-family) or **unimodal** (uniformly busy / uniformly quiet) patterns, with per-tile `busy=True/False` labels and busy-restricted surface coverage statistics.

Operates on the v3.1 Planet Four catalog. Methodology: zero-inflation-aware GMM-BIC bimodality test on `log1p` of per-tile marking counts; learned global threshold from confidently-bimodal split-points; coverage statistics restricted to `busy=True` tiles for use by climate modellers needing a number for the active portions of the seasonal cap.

In [ ]:
#| export
"""classify_by_activity — tile-activity classification for the v3.1 Planet Four catalog.

For each HiRISE observation in the v3.1 catalog this module decides whether
the per-tile marking-count distribution is **bimodal** (busy / not-busy
two-family) or **unimodal** (uniformly busy or uniformly quiet) and labels
each tile ``busy=True/False``. It then computes the fractional surface
coverage (e.g. from ``FnotchCoverage_Full_v3.1.csv``) restricted to the
busy tiles, providing a more informative number for climate modellers
than the obsid-mean coverage.

Methodology (zero-inflation-aware):

1. Per-tile marking count = ``#fan rows + #blotch rows`` for that tile.
   The per-tile coverage table is the authoritative tile inventory;
   missing tiles get ``count = 0``.
2. For each obsid, fit ``sklearn.mixture.GaussianMixture`` with 1 and 2
   components on ``log1p(counts)`` and compute BIC for each.
3. Classify the obsid:

   * fewer than ``MIN_TILES`` (default 20) tiles → forced unimodal,
     busy/quiet by mean count vs the global threshold.
   * ``frac_zero >= FRAC_ZERO_HIGH`` → ``unimodal_quiet``.
   * ``frac_zero <= FRAC_ZERO_LOW`` and weak BIC contrast → ``unimodal_busy``.
   * Otherwise → ``bimodal``.

4. Per-tile ``busy``:

   * **bimodal** — hard-assignment to the higher-mean GMM component.
   * **unimodal_busy** — all True.
   * **unimodal_quiet** — all False.

5. The global threshold T is learned in a first pass as the median of
   the bimodal-obsid GMM split-points (in raw count units).

The fan and blotch catalogs are loaded by default through
:func:`p4tools.io.get_fan_catalog` and :func:`p4tools.io.get_blotch_catalog`,
so the module works for any p4tools user without hardcoded local paths.
The per-tile coverage table is supplied separately (it is currently an
external product not yet shipped with the public catalog).
"""
from dataclasses import dataclass, field
from enum import StrEnum
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture

from p4tools.io import get_blotch_catalog, get_fan_catalog


class Pattern(StrEnum):
    """Classification of an obsid's tile-count distribution."""
    BIMODAL = "bimodal"
    UNIMODAL_BUSY = "unimodal_busy"
    UNIMODAL_QUIET = "unimodal_quiet"


@dataclass(frozen=True)
class ClassifyConfig:
    """Decision thresholds for the per-obsid classifier.

    Attributes
    ----------
    delta_bic
        Minimum BIC(1)-BIC(2) for declaring bimodal (Kass & Raftery 1995
        "strong evidence" cut, default 10.0).
    min_tiles
        Below this tile count GMM(2) is unstable; obsid is forced unimodal.
    frac_zero_high
        Above this zero-fraction the obsid is mostly empty → unimodal_quiet.
    frac_zero_low
        Below this zero-fraction the obsid is uniformly active candidate.
    random_state
        Seed for GaussianMixture initialisation (determinism).
    global_threshold
        Raw-count threshold applied only to forced-unimodal obsids
        (n_tiles < min_tiles). ``None`` triggers learning from the
        bimodal split-points in :func:`classify_all`.
    """
    delta_bic: float = 10.0
    min_tiles: int = 20
    frac_zero_high: float = 0.85
    frac_zero_low: float = 0.15
    random_state: int = 42
    global_threshold: float | None = None


DEFAULT_CONFIG = ClassifyConfig()

## Data loading

Defaults to the public v3.1 fan/blotch parquets via `p4tools.io`. The per-tile coverage CSV is external and supplied as an explicit path.

In [ ]:
#| export
@dataclass
class P4Data:
    """Container for the three v3.1 dataframes used by this module."""
    fan: pd.DataFrame
    blotch: pd.DataFrame
    coverage: pd.DataFrame


def load_v3p1_data(
    coverage_csv: str | Path,
    *,
    version: str = "v3.1",
    fan: pd.DataFrame | None = None,
    blotch: pd.DataFrame | None = None,
) -> P4Data:
    """Load the v3.1 fan, blotch, and per-tile coverage frames.

    Parameters
    ----------
    coverage_csv
        Path to a CSV with columns ``[obsid, tile_id, Coverage]``
        (e.g. ``FnotchCoverage_Full_v3.1.csv``).
    version
        Catalog version forwarded to ``p4tools.io.get_*_catalog``.
    fan, blotch
        Optional pre-loaded dataframes; useful in tests or when an
        offline copy is preferred over the pooch-cached parquet.
    """
    fan_df = fan if fan is not None else get_fan_catalog(version)
    blotch_df = blotch if blotch is not None else get_blotch_catalog(version)
    coverage = pd.read_csv(coverage_csv)
    expected = {"obsid", "tile_id", "Coverage"}
    missing = expected.difference(coverage.columns)
    if missing:
        raise ValueError(f"coverage CSV is missing columns: {missing}")
    return P4Data(
        fan=fan_df[["tile_id", "obsid"]].copy(),
        blotch=blotch_df[["tile_id", "obsid"]].copy(),
        coverage=coverage[list(expected)].copy(),
    )

## Per-tile marking counts

In [ ]:
#| export
def _count_by_tile(df: pd.DataFrame, name: str) -> pd.DataFrame:
    return (
        df.groupby(["obsid", "tile_id"], sort=False)
        .size()
        .rename(name)
        .reset_index()
    )


def count_markings_per_tile(data: P4Data) -> pd.DataFrame:
    """One row per (obsid, tile_id) with total marking counts.

    Tiles present in ``data.coverage`` but absent from the catalogs
    appear with ``n_markings = 0``. Returned columns:
    ``[obsid, tile_id, n_fans, n_blotches, n_markings, Coverage]``.
    """
    fan_counts = _count_by_tile(data.fan, "n_fans")
    blotch_counts = _count_by_tile(data.blotch, "n_blotches")
    out = (
        data.coverage.merge(fan_counts, on=["obsid", "tile_id"], how="left")
        .merge(blotch_counts, on=["obsid", "tile_id"], how="left")
    )
    out[["n_fans", "n_blotches"]] = out[["n_fans", "n_blotches"]].fillna(0).astype(int)
    out["n_markings"] = out["n_fans"] + out["n_blotches"]
    return out[["obsid", "tile_id", "n_fans", "n_blotches", "n_markings", "Coverage"]]

## Per-obsid classification

In [ ]:
#| export
@dataclass
class ObsidClassification:
    """Result of classifying a single HiRISE observation's tile distribution."""
    obsid: str
    pattern: Pattern
    n_tiles: int
    frac_zero: float
    mean_count: float
    bic1: float
    bic2: float
    cluster_means: tuple[float, float] | None  # (low, high) in log1p-space
    busy_mask: np.ndarray  # bool array aligned with the input counts

    @property
    def delta_bic(self) -> float:
        if not (np.isfinite(self.bic1) and np.isfinite(self.bic2)):
            return float("nan")
        return self.bic1 - self.bic2

    @property
    def split_point_log(self) -> float | None:
        """Bimodal split-point in log1p-space (mean of the two cluster means)."""
        if self.cluster_means is None:
            return None
        return 0.5 * (self.cluster_means[0] + self.cluster_means[1])

    @property
    def split_point_count(self) -> float | None:
        """Bimodal split-point converted back to raw count units."""
        sp = self.split_point_log
        return None if sp is None else float(np.expm1(sp))


def _fit_gmm_bic(values: np.ndarray, n_components: int, random_state: int):
    try:
        model = GaussianMixture(
            n_components=n_components,
            random_state=random_state,
            covariance_type="full",
            reg_covar=1e-6,
            max_iter=200,
        ).fit(values.reshape(-1, 1))
        return model, model.bic(values.reshape(-1, 1))
    except Exception:
        return None, float("nan")


def classify_obsid(
    counts: np.ndarray,
    obsid: str = "",
    *,
    config: ClassifyConfig = DEFAULT_CONFIG,
    global_threshold: float | None = None,
) -> ObsidClassification:
    """Classify one obsid's tile-count distribution.

    ``global_threshold`` overrides ``config.global_threshold`` for the
    forced-unimodal small-obsid path.
    """
    counts = np.asarray(counts, dtype=float)
    n_tiles = counts.size
    if n_tiles == 0:
        raise ValueError(f"obsid {obsid!r} has no tiles")
    threshold = global_threshold if global_threshold is not None else (
        config.global_threshold if config.global_threshold is not None else 1.0
    )
    frac_zero = float((counts == 0).mean())
    mean_count = float(counts.mean())
    log_counts = np.log1p(counts)

    if n_tiles < config.min_tiles:
        m1, bic1 = _fit_gmm_bic(log_counts, 1, config.random_state)
        bic2 = float("nan")
        m2 = None
        pattern = Pattern.UNIMODAL_BUSY if mean_count > threshold else Pattern.UNIMODAL_QUIET
    else:
        m1, bic1 = _fit_gmm_bic(log_counts, 1, config.random_state)
        m2, bic2 = _fit_gmm_bic(log_counts, 2, config.random_state)
        delta = (bic1 - bic2) if (np.isfinite(bic1) and np.isfinite(bic2)) else float("nan")
        if frac_zero >= config.frac_zero_high:
            pattern = Pattern.UNIMODAL_QUIET
        elif frac_zero <= config.frac_zero_low and (
            not np.isfinite(delta) or delta < config.delta_bic
        ):
            pattern = Pattern.UNIMODAL_BUSY
        else:
            pattern = Pattern.BIMODAL

    cluster_means: tuple[float, float] | None = None
    busy_mask = np.zeros(n_tiles, dtype=bool)

    if pattern is Pattern.BIMODAL and m2 is not None:
        means = m2.means_.flatten()
        order = np.argsort(means)
        low_mean = float(means[int(order[0])])
        high_mean = float(means[int(order[1])])
        cluster_means = (low_mean, high_mean)
        labels = m2.predict(log_counts.reshape(-1, 1))
        busy_mask = labels == int(order[1])
    elif pattern is Pattern.BIMODAL:
        # GMM(2) failed but the heuristic flagged bimodal — fall back to
        # the natural zero/nonzero boundary.
        busy_mask = counts > 0
    elif pattern is Pattern.UNIMODAL_BUSY:
        busy_mask[:] = True

    return ObsidClassification(
        obsid=obsid,
        pattern=pattern,
        n_tiles=n_tiles,
        frac_zero=frac_zero,
        mean_count=mean_count,
        bic1=float(bic1) if np.isfinite(bic1) else float("nan"),
        bic2=float(bic2) if np.isfinite(bic2) else float("nan"),
        cluster_means=cluster_means,
        busy_mask=busy_mask,
    )

## All-obsid classification

First pass: classify every obsid; gather bimodal split-points to learn the global threshold T. Second pass: only the small-obsid forced-unimodal entries are re-classified with T.

In [ ]:
#| export
def classify_all(
    tile_counts: pd.DataFrame,
    *,
    config: ClassifyConfig = DEFAULT_CONFIG,
) -> tuple[pd.DataFrame, dict]:
    """Classify every obsid and label every tile.

    Returns
    -------
    labeled
        ``[obsid, tile_id, n_fans, n_blotches, n_markings, Coverage,
        pattern, busy]``.
    params
        Dict of the parameters actually used (incl. learned ``global_threshold``).
    """
    required = {"obsid", "tile_id", "n_markings", "Coverage"}
    missing = required.difference(tile_counts.columns)
    if missing:
        raise ValueError(f"tile_counts is missing columns: {missing}")

    obsid_groups = list(tile_counts.groupby("obsid", sort=False))
    pass1: dict[str, ObsidClassification] = {}
    for obsid, sub in obsid_groups:
        pass1[obsid] = classify_obsid(
            sub["n_markings"].to_numpy(),
            obsid=obsid,
            config=config,
            global_threshold=1.0,  # placeholder; refined below
        )

    if config.global_threshold is None:
        split_pts = [
            r.split_point_count for r in pass1.values()
            if r.pattern is Pattern.BIMODAL and r.split_point_count is not None
        ]
        learned = float(np.median(split_pts)) if split_pts else 1.0
    else:
        learned = config.global_threshold

    rows = []
    for obsid, sub in obsid_groups:
        result = pass1[obsid]
        if result.n_tiles < config.min_tiles:
            # Re-run only the small obsids; T may have flipped their label.
            result = classify_obsid(
                sub["n_markings"].to_numpy(),
                obsid=obsid,
                config=config,
                global_threshold=learned,
            )
        sub = sub.copy()
        sub["pattern"] = result.pattern.value
        sub["busy"] = result.busy_mask
        rows.append(sub)

    labeled = pd.concat(rows, ignore_index=True)
    params = dict(
        global_threshold=float(learned),
        delta_bic=config.delta_bic,
        min_tiles=config.min_tiles,
        frac_zero_high=config.frac_zero_high,
        frac_zero_low=config.frac_zero_low,
        random_state=config.random_state,
    )
    return labeled, params

## Coverage on busy tiles

Vectorised aggregation across the cap (no Python-level loop over obsids).

In [ ]:
#| export
def coverage_on_busy(labeled: pd.DataFrame) -> pd.DataFrame:
    """Per-obsid coverage statistics, split by busy/quiet."""
    base = labeled.groupby("obsid", sort=False).agg(
        pattern=("pattern", "first"),
        n_tiles=("busy", "size"),
        n_busy=("busy", "sum"),
        all_median=("Coverage", "median"),
        all_mean=("Coverage", "mean"),
    )

    def _split(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
        if df.empty:
            return pd.DataFrame(
                index=labeled["obsid"].unique(),
                columns=[f"{prefix}_median", f"{prefix}_mean", f"{prefix}_q25", f"{prefix}_q75"],
            )
        agg = df.groupby("obsid", sort=False)["Coverage"].agg(
            median="median",
            mean="mean",
            q25=lambda s: s.quantile(0.25),
            q75=lambda s: s.quantile(0.75),
        )
        agg.columns = [f"{prefix}_{c}" for c in agg.columns]
        return agg

    busy_stats = _split(labeled[labeled["busy"]], "busy")
    quiet_stats = _split(labeled[~labeled["busy"]], "quiet")

    return base.join(busy_stats).join(quiet_stats).reset_index()


def summary(labeled: pd.DataFrame, params: dict) -> dict:
    """Cap-wide one-line summary suitable for printing or logging."""
    pattern_counts = labeled.drop_duplicates("obsid")["pattern"].value_counts().to_dict()
    busy = labeled[labeled["busy"]]
    return dict(
        n_obsids=int(labeled["obsid"].nunique()),
        n_tiles=int(len(labeled)),
        n_busy_tiles=int(busy.shape[0]),
        n_bimodal=int(pattern_counts.get(Pattern.BIMODAL.value, 0)),
        n_unimodal_busy=int(pattern_counts.get(Pattern.UNIMODAL_BUSY.value, 0)),
        n_unimodal_quiet=int(pattern_counts.get(Pattern.UNIMODAL_QUIET.value, 0)),
        cap_busy_median_coverage=float(busy["Coverage"].median()) if len(busy) else float("nan"),
        cap_busy_mean_coverage=float(busy["Coverage"].mean()) if len(busy) else float("nan"),
        cap_all_median_coverage=float(labeled["Coverage"].median()),
        cap_all_mean_coverage=float(labeled["Coverage"].mean()),
        learned_global_threshold=params["global_threshold"],
    )

## CLI

End-to-end pipeline writing per-tile labels and per-obsid summary CSVs.

In [ ]:
#| export
def run_pipeline(
    coverage_csv: str | Path,
    *,
    out_dir: str | Path = "outputs",
    config: ClassifyConfig = DEFAULT_CONFIG,
    version: str = "v3.1",
) -> dict:
    """Run the full classification pipeline and write CSVs to ``out_dir``."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    data = load_v3p1_data(coverage_csv, version=version)
    counts = count_markings_per_tile(data)
    labeled, params = classify_all(counts, config=config)
    per_obsid = coverage_on_busy(labeled)
    s = summary(labeled, params)

    labeled.to_csv(out_dir / f"tiles_labeled_{version}.csv", index=False)
    per_obsid.to_csv(out_dir / f"per_obsid_busy_coverage_{version}.csv", index=False)
    pd.Series(params).to_csv(out_dir / f"classification_params_{version}.csv", header=["value"])
    pd.Series(s).to_csv(out_dir / f"cap_summary_{version}.csv", header=["value"])
    return s


def main() -> None:  # pragma: no cover
    """CLI entry; prints the cap-wide summary."""
    import argparse
    parser = argparse.ArgumentParser(
        description="Classify v3.1 P4 tiles into busy/quiet and report busy-restricted coverage."
    )
    parser.add_argument("--coverage-csv", required=True,
                        help="path to FnotchCoverage_Full_v3.1.csv (or compatible)")
    parser.add_argument("--out-dir", default="outputs")
    parser.add_argument("--version", default="v3.1")
    parser.add_argument("--delta-bic", type=float, default=DEFAULT_CONFIG.delta_bic)
    parser.add_argument("--min-tiles", type=int, default=DEFAULT_CONFIG.min_tiles)
    parser.add_argument("--frac-zero-high", type=float, default=DEFAULT_CONFIG.frac_zero_high)
    parser.add_argument("--frac-zero-low", type=float, default=DEFAULT_CONFIG.frac_zero_low)
    parser.add_argument("--random-state", type=int, default=DEFAULT_CONFIG.random_state)
    args = parser.parse_args()
    config = ClassifyConfig(
        delta_bic=args.delta_bic,
        min_tiles=args.min_tiles,
        frac_zero_high=args.frac_zero_high,
        frac_zero_low=args.frac_zero_low,
        random_state=args.random_state,
    )
    s = run_pipeline(
        coverage_csv=args.coverage_csv,
        out_dir=args.out_dir,
        config=config,
        version=args.version,
    )
    for k, v in s.items():
        print(f"{k:32s} {v}")

---

Manual entry point so `python -m p4tools.classify_by_activity --coverage-csv ...` works:

In [ ]:
#| export
if __name__ == "__main__":
    main()